# CSID parameter estimation (minimal example)

Estimate **CSID-2** pair coefficients (`J_hat`) and **CSID-3** triple coefficients (`theta_hat`) on basket data.

Run from the repo root after `uv sync --group dev`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "csid").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from csid.analysis import AnalysisConfig, fit_generated_candidates
from csid.data import SparseBasketDataset
from csid.estimator import fit_to_dataframe

## 1. Example data

Six grocery categories, 500 baskets, random sizes 2–4.
Milk and cereal co-occur more often than chance.

Each row of the internal matrix is one basket; each column is a category (0/1 presence).

In [2]:
CATEGORIES = ["milk", "cereal", "fruit", "bread", "eggs", "yogurt"]
rng = np.random.default_rng(42)
n, c = 500, len(CATEGORIES)

X = np.zeros((n, c), dtype=np.uint8)
for row in range(n):
    items = rng.choice(c, size=int(rng.integers(2, 5)), replace=False)
    if rng.random() < 0.35:
        items = np.unique(np.concatenate([items, [0, 1]]))  # milk + cereal
    X[row, items] = 1

dataset = SparseBasketDataset.from_dense(
    X,
    name="grocery-demo",
    category_names=CATEGORIES,
)

print(f"{dataset.n_baskets} baskets × {dataset.n_categories} categories")
print(f"mean basket size: {dataset.basket_sizes.mean():.2f}")

500 baskets × 6 categories
mean basket size: 3.32


## 2. Fit CSID

`fit_generated_candidates` builds rest-cardinality contrasts and fits gauge-constrained ridge CSID-2/3 on **all baskets**.

In [3]:
config = AnalysisConfig(
    candidate_mode="exhaustive",  # fine for small category sets
    triple_min_count=0,
    min_rest_size=0,
    epsilon=0.5,
    ridge2=1.0,
    ridge3=1.0,
)

bundle = fit_generated_candidates(
    dataset,
    config=config,
    cache_dir=ROOT / "notebooks" / "outputs" / "cache",
)

print(f"CSID-2: {len(bundle.fit2.theta)} pairs, converged={bundle.fit2.converged}")
print(f"CSID-3: {len(bundle.fit3.theta)} triples, converged={bundle.fit3.converged}")

CSID-2: 15 pairs, converged=True
CSID-3: 20 triples, converged=True


## 3. Results

In [4]:
support = np.asarray(dataset.X.mean(axis=0)).ravel()

pairs = fit_to_dataframe(bundle.fit2, category_names=CATEGORIES, singleton_support=support)
triples = fit_to_dataframe(bundle.fit3, category_names=CATEGORIES, singleton_support=support)

print("Strongest pairs (CSID-2):")
display(pairs.nlargest(5, "J_hat")[["item_1", "item_2", "J_hat", "z"]])

print("Strongest |z| triples (CSID-3):")
display(
    triples.reindex(triples["z"].abs().sort_values(ascending=False).index)
    .head(5)[["item_1", "item_2", "item_3", "theta_hat", "z"]]
)

Strongest pairs (CSID-2):


,item_1,item_2,J_hat,z
0,milk,cereal,1.213241,5.365261
5,cereal,fruit,0.212949,1.009977
1,milk,fruit,0.129538,0.635395
8,cereal,yogurt,0.026272,0.123324
6,cereal,bread,0.024308,0.112483


Strongest |z| triples (CSID-3):


,item_1,item_2,item_3,theta_hat,z
14,cereal,bread,yogurt,-0.713213,-1.544806
2,milk,cereal,eggs,0.567435,1.171177
16,fruit,bread,eggs,-0.498718,-1.166372
4,milk,fruit,bread,0.426465,0.982888
12,cereal,fruit,yogurt,-0.377747,-0.845908


## 4. Using your own CSV (optional)

Long format: one row per `(basket, item)` pair.

```python
from csid.data import load_generic

dataset = load_generic(
    "your_transactions.csv",
    basket_col="basket_id",
    item_col="category",
    period_col="period",
    top_n=20,
)
bundle = fit_generated_candidates(dataset, config=config, cache_dir="cache")
```

CLI equivalent:

```bash
uv run csid analyze --dataset generic --input your.csv \
  --basket-col basket_id --item-col category --period-col period \
  --top-n 20 --output outputs/run
```